# Features: from an idea to evidence

The demo of Phase 3. You write a feature, check it with the leak detector, compute it through the feature cache, build a second feature from others with the combinators, and judge both with `ff.analytics` before any strategy exists: their forward returns by quantile, their daily rank information coefficient and how it decays with the horizon, whether they say the same, and what follows the bars where a flag turns true. Every result ends with the trials counter, how many features and horizons you have examined so far, so that a promising number is read against how much was tried to find it. `docs/engine-guide.md` explains every table, in "Writing a feature" and "Judging a feature".

Pick the `.venv` kernel and run all cells. The notebook reads the store in `FEFAT_DATA_DIR` and keeps the feature cache and your library under `FEFAT_HOME`. It prints no price: the forward returns are in ticks, and the features are ratios.

## A feature of your own

The product is MES when your store holds its 1-minute bars, else the product with the most contracts, and the window the three months to its last trade date. `close_location` says where the close lies in the range of the last 20 bars: 0 at their lowest low, 1 at their highest high, and no value where they have not moved. It is a ratio, so it reads the same at any price.

Expected: the product and the window, and the feature's hash, which its code and parameters decide.

In [ ]:
import time
from datetime import timedelta

import matplotlib.pyplot as plt
import polars as pl

import fefat as ff
from fefat.features import combine

listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
if listed.is_empty():
    raise ValueError(
        "Expected 1-minute bars of at least one product in your store, got none: import your "
        "files first (docs/data-guide.md)."
    )
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, LAST = chosen["root"], chosen["last_trade_date"]
START = LAST - timedelta(days=91)


@ff.feature(params={"n": 20}, warmup=lambda n: n - 1, outputs={"close_location": "ratio"})
def close_location(bars: pl.LazyFrame, n: int) -> pl.Expr:
    highest, lowest = pl.col("high").rolling_max(n), pl.col("low").rolling_min(n)
    return pl.when(highest > lowest).then((pl.col("close") - lowest) / (highest - lowest))


print(f"{ROOT} 5-minute bars from {START} to {LAST}")
print(f"close_location, hash {close_location.hash[:12]}")

## The leak detector

`ff.features.validate` computes the feature on the last two weeks, then again on the data cut at many instants, and requires the values before each cut to stay the same: a feature that reads ahead changes them.

Expected: `PASSED`, with no cut failing and no finding of the lint. A feature that fails stops the notebook here, since only a feature that reads no future is worth judging.

In [ ]:
CHECKED_FROM = LAST - timedelta(days=14)
report = ff.features.validate(close_location(), ROOT, "5m", CHECKED_FROM, LAST)
print(report.summary())
if not report.passed:
    raise ValueError(f"Expected close_location to read no future, got {report.first_failure()}")
print(f"The leak detector on {ROOT}'s 5-minute bars from {CHECKED_FROM}: PASSED")

## A combination, computed through the cache

`rsi_stretch` is RSI(14) less its mean over ten bars, built with the combinators: a feature as any other. `ff.features.compute` evaluates the three features over the window and keeps their values in the feature cache, so the second call reads them back. `oversold`, RSI under 30, is a flag for the event study below.

Expected: the bars of the window, and the two calls' times, the second much the shorter, with the same values.

In [ ]:
rsi = ff.indicators.rsi
rsi_stretch = combine.define("rsi_stretch", rsi(14) - combine.rolling_mean(rsi(14), 10))
SLOTS = {"close_location": close_location(), "rsi": rsi(14), "rsi_stretch": rsi_stretch()}


def timed_compute() -> tuple[pl.DataFrame, float]:
    began = time.perf_counter()
    computed = ff.features.compute(SLOTS, ROOT, "5m", START, LAST).collect()
    return computed, time.perf_counter() - began


frame, computing = timed_compute()
again, reading = timed_compute()
print(f"{frame.height:,} bars, {frame['zero_volume'].sum():,} of them without trades")
print(f"first call {computing:.2f} s, second call {reading:.2f} s")
print("the second read the same values:", again.equals(frame))
frame = frame.with_columns(oversold=pl.col("rsi") < 30)

## Forward returns by quantile

A study labels every bar with its forward returns: what a market order decided at the bar's close makes when held 1, 5, or 20 bars, from the next bar's open to the open after the last bar held, in ticks, within the trade date. It sorts the bars into five quantiles of the feature and gives each quantile's mean return with a 95 % interval from resampling whole trade dates. Each interval is 95 % on its own: among the fifteen, one that leaves 0 by chance alone is more likely than not.

Expected, your check of this step: the table of `rsi_stretch`. If the feature has an edge, the extreme quantiles differ from the middle ones; where an interval holds 0, the data cannot tell that quantile's mean from nothing.

In [ ]:
study = ff.analytics.FeatureStudy(frame, ROOT, horizons=(1, 5, 20))
quantiles = study.quantile_returns("rsi_stretch")
print(quantiles.report())

The same table at the longest horizon as a plot: each quantile's mean with its interval.

Expected: five points with their bars; a feature with an edge slopes, and a bar crossing the grey line at 0 is no evidence either way.

In [ ]:
longest = quantiles.frame.filter(pl.col("horizon") == 20)
mean = longest.get_column("mean_ticks")
below = mean - longest.get_column("interval_low_ticks")
above = longest.get_column("interval_high_ticks") - mean
figure, axes = plt.subplots(figsize=(7, 3.5))
axes.errorbar(longest.get_column("quantile"), mean, yerr=[below, above], fmt="o", capsize=4)
axes.axhline(0, color="grey", linewidth=0.8)
axes.set_xlabel("quantile of rsi_stretch")
axes.set_ylabel("mean return over 20 bars, ticks")
axes.set_title(f"{ROOT}: the returns after each quantile of rsi_stretch")
plt.show()

## The information coefficient

Each trade date's share of the feature's rank correlation with the returns after it, both ranked over the whole window, its mean over the dates, and its t-statistic, each day one piece of evidence. A t-statistic beyond about 2 is unlikely by chance for one feature at one horizon; the trials line under each table raises that bar for all you examined. The mean usually falls as the horizon grows: the decay, how long what the feature foretells lasts.

Expected: a table for `rsi_stretch` and one for `close_location`, by horizon and by year, and the trials line growing from one feature to two.

In [ ]:
print(study.information_coefficient("rsi_stretch").report())
print()
print(study.information_coefficient("close_location").report())

## Do they say the same?

The rank correlations of the features with each other. Features that correlate at 0.7 or more, either way, share a cluster: a strategy rarely needs two of them. `rsi_stretch` is built from `rsi`, so see how much of it the RSI already says. A correlation of features compares no return, so the trials stay as they were.

Expected: the clusters and the matrix.

In [ ]:
print(study.correlations(["close_location", "rsi", "rsi_stretch"]).report())

## What follows a flag

An event study of `oversold` for a long: an event is a bar where the flag turns true, and the path is the mean and median return 1 to 20 bars after it. The excursions say how far the price went against the position and for it while held, its MAE and MFE, which is where a stop and a target would sit. Following the flag over 20 bars examines 20 horizons, which the trials count.

Expected: the events, the path, and the excursions.

In [ ]:
print(study.event_study("oversold", ff.analytics.PositionSide.LONG, horizon=20).report())

## The trials counter

Every analysis above counted the features and horizons it examined. The line says how many, and how far from 0 a t-statistic must now lie to keep 5 % significance across all of them: the more you look, the more a lucky number appears, and the higher the bar.

Expected: three features and twenty horizons examined.

In [ ]:
print(study.trials.line())

## Reading the results

- **A quantile table** is evidence where its intervals leave 0 and its means move in order from the first quantile to the last. One quantile apart from the rest, with its neighbours flat, is more often noise than an edge.
- **The information coefficient** of a useful intraday feature is small, often 0.01 to 0.05; what matters is its t-statistic against the trials line's bar, and that it holds year by year.
- **A cluster** of features says one thing: keep the one you understand best.
- **An event study's excursions** show how far a position goes against you before it pays: a stop inside the typical MAE stops most trades out.
- **The trials counter** never resets inside a study. Start a new study and it starts again, so keep one study for one line of research, and note the count beside any result you keep.